# Day 11 · 裁判验证协议：从检验到上线门槛

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 写出一份可执行的裁判验证协议，包括事先确定的上线门槛
- 以咨询师间一致性作为裁判一致性的参照上限
- 分层报告（人群、风险、维度），发现总体一致掩盖的问题
- 用代码生成一份裁判验证报告

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 准备数据

In [2]:
from course.data import simulate_counselor_study, DIMENSIONS
from course.agreement import krippendorff_alpha, cohen_kappa, icc, bootstrap_ci
study = simulate_counselor_study(seed=0)
rep, h, j = study["replies"], study["human"], study["judge"]
pop = rep.set_index(["item", "system"]).population

## 2. 人类参照：咨询师间一致性（每个维度）

In [3]:
rows = []
for d in DIMENSIONS:
    m = h[h.dim == d].pivot_table(index=["item", "system"], columns="rater", values="score")
    rows.append({"dim": d, "alpha_counselors": krippendorff_alpha(m.values.T, "ordinal")})
human_ref = pd.DataFrame(rows).set_index("dim"); human_ref

,alpha_counselors
dim,
D1,0.653
D2,0.592
D3,0.756
D4,0.593
D5,0.551


## 3. 裁判与咨询师：二次加权 kappa

咨询师均值四舍五入到整数后，与裁判第 0 次评分比较。

In [4]:
def judge_vs_human(d, subset=None):
    hm = h[h.dim == d].groupby(["item", "system"]).score.mean().round().astype(int)
    jm = j[(j.dim == d) & (j.run == 0)].set_index(["item", "system"]).score
    idx = hm.index if subset is None else hm.index[subset(hm.index)]
    return bootstrap_ci(cohen_kappa, hm.loc[idx].values, jm.loc[idx].values, n_boot=800,
                        weights="quadratic", categories=[1, 2, 3, 4, 5])
judge_vs_human("D1")

(0.38956521739130434, 0.2782279158766941, 0.49144386520669936)

**练习 11.1**：分人群计算 D3（引导发现）上的裁判一致性：老年人题和其他题分别给出 kappa 点估计。填入老年人题的点估计。

In [5]:
is_older = lambda idx: np.array([pop.loc[i] == "older adult" for i in idx])
older = judge_vs_human("D3", is_older)
others = judge_vs_human("D3", lambda idx: ~is_older(idx))
print("老年人:", np.round(older, 3), " 其他:", np.round(others, 3))
ans_11_1 = None
check("练习 11.1 老年人题 D3 kappa", ans_11_1, older[0], tol=1e-6)

老年人: [0.491 0.372 0.603]  其他: [0.408 0.284 0.537]
[ 待完成 ] 练习 11.1 老年人题 D3 kappa


## 4. 重复采样：裁判跨次一致性

In [6]:
runs = j[j.dim == "D1"].pivot_table(index=["item", "system"], columns="run", values="score")
print("跨 3 次的 ICC(2,1):", round(icc(runs.values, "ICC(2,1)"), 3))

跨 3 次的 ICC(2,1): 0.849


## 5. 写出 judge_report

**练习 11.2**：完成 `judge_report()`，对每个维度返回一行：`alpha_counselors`、`kappa`、`kappa_lo`、`kappa_hi`、`retest_icc`，以及 `pass`（规则：`kappa_lo >= alpha_counselors - 0.1` 且 `retest_icc >= 0.8`）。

In [7]:
def judge_report():
    rows = []
    for d in DIMENSIONS:
        row = {"dim": d}
        # 在这里填写:alpha_counselors, kappa, kappa_lo, kappa_hi, retest_icc, pass
        rows.append(row)
    return pd.DataFrame(rows).set_index("dim")

rep_tab = judge_report(); rep_tab

""
dim
D1
D2
D3
D4
D5


In [8]:
_ok = all(c in rep_tab.columns for c in ["alpha_counselors", "kappa", "kappa_lo", "kappa_hi", "retest_icc", "pass"])
check_true("练习 11.2 报告包含全部列", _ok if _ok else None)
if _ok:
    _exp_pass = [bool(r.kappa_lo >= r.alpha_counselors - 0.1 and r.retest_icc >= 0.8) for r in rep_tab.itertuples()]
    check("练习 11.2 pass 列与规则一致", list(rep_tab["pass"]), _exp_pass)

[ 待完成 ] 练习 11.2 报告包含全部列


**练习 11.3**：根据报告，写三句话的结论：这个裁判可以用于哪些维度？不能用于哪些？需要补充什么证据？写完把 `ans_11_3` 改成 `True`。

*结论：*

In [9]:
ans_11_3 = None
check_true("练习 11.3 写完结论", ans_11_3)

[ 待完成 ] 练习 11.3 写完结论


## 6. （可选）接入真实 LLM 裁判

设置好 `LLM_API_KEY` 等环境变量后运行。没有设置时这一格会跳过。

In [10]:
from course.llm import get_client
from course.judge import LLMJudge
from course.data import RUBRIC_TEXT, load_prompts, template_reply
client = get_client()
if client is None:
    print("未设置 LLM_API_KEY,跳过真实裁判。")
else:
    p = load_prompts().iloc[0]
    judge = LLMJudge(client, RUBRIC_TEXT)
    print(judge.rate(p.prompt, template_reply("empathic", p.topic)))

未设置 LLM_API_KEY,跳过真实裁判。


## 口述题（录音，每题 90 秒）

1. 你会怎么设计一个裁判的上线门槛？
   <details><summary>参考要点</summary>以咨询师间一致性为参照设门槛；重复采样信度；位置互换翻转率；长度和自我偏好的偏差检验；分维度、分人群、分风险报告；版本锁定和锚定集复测。门槛事先写定。</details>
2. 裁判总体 κ 达标，但在老年人题上明显偏低，怎么办？
   <details><summary>参考要点</summary>不能直接上线到老年人场景。报告分层结果，扩充老年人题和咨询师评分，查明原因（语言风格、文化差异、题目难度），必要时为该人群单独校准或保留人工评分。</details>

## 研究延伸

一个完整的裁判验证协议，加上咨询师作为金标准，本身就是可发表的方法学贡献。可以把协议事先公开（类似预注册），再报告结果，包括裁判不合格的维度。

In [11]:
summary()

本 notebook 自动批改:0 / 3 通过
